# TopoStats - Minicircle Walk-Through

Welcome, this [Jupyter Notebook](https://jupyter.org/) will take you through processing `minicircle.spm` - a nanoscale AFM height image of DNA atop a flat mica surface.

## Installing TopoStats

There are several different ways of installing TopoStats depending on what you want to do. The simplest is to install
from GitHub under a virtual environment.

```bash
pip install git+https://github.com/AFM-SPM/TopoStats.git@main
```

For more information on the different ways of installing TopoStats and setting up Virtual Environments please refer to
[installation](https://afm-spm.github.io/TopoStats/installation.html).



## Getting Started


### Loading Libraries and Modules

TopoStats is written as a series of modules with various classes and functions. In order to use these interactively we
need to `import` them.

In [ ]:
import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

from topostats.filters import Filters
from topostats.grains import Grains
from topostats.grainstats import GrainStats
from topostats.io import LoadScans, find_files, read_yaml
from topostats.plottingfuncs import Images
from topostats.processing import get_out_paths, run_grains
from topostats.tracing.disordered_tracing import trace_image_disordered

## Finding Files

When run from the command line TopoStats needs to find files to process and the `find_files()` function helps here. It
takes as an argument the directory path that should be searched and the file extension to look for (this example uses `.spm`
files) and returns a list of all files in the specified directory which have that file extension
directory. We can use that functionality in this Notebook if you place your files in the same directory as these
Notebooks and execute the next cell.

In [ ]:
# Set the base directory to be current working directory of the Notebook
BASE_DIR = Path().cwd()
# Alternatively if you know where your files area comment the above line and uncomment the below adjust it for your use.
# BASE_DIR = Path("/path/to/where/my/files/are")
# Adjust the file extension appropriately.
FILE_EXT = ".spm"
# Search for *.spm files one directory level up from the current notebooks
image_files = find_files(base_dir=BASE_DIR.parent / "tests" / "resources", file_ext=FILE_EXT)

`image_files` is a list of images that match and we can look at that list.

In [ ]:
image_files

## Loading a Configuration

You can specify all options explicitly by hand when instantiating classes or calling methods/functions. However when run
at the command line in batch mode TopoStats loads these options from a [YAML](https://yaml.org/) configuration file and it is worth
understanding the structure of this file and how it is used.

A trimmed version is shown below. The words that come before the colon `:` are the option, e.g. `base_dir:` is the base
directory that is searched for files, what comes after is the value, in this case `./tests/`


```yaml
base_dir: ./ # Directory in which to search for data files
output_dir: ./output # Directory to output results to
log_level: info # Verbosity of output. Options: warning, error, info, debug
cores: 2 # Number of CPU cores to utilise for processing multiple files simultaneously.
file_ext: .spm # File extension of the data files.
loading:
  channel: Height # Channel to pull data from in the data files.
filter:
  run: true # Options : true, false
  row_alignment_quantile: 0.5 # below values may improve flattening of larger features
  threshold_method: std_dev # Options : otsu, std_dev, absolute
  otsu_threshold_multiplier: 1.0
  threshold_std_dev:
    below: 10.0 # Threshold for data below the image background
    above: 1.0 # Threshold for data above the image background
  threshold_absolute:
    below: -1.0 # Threshold for data below the image background
    above: 1.0 # Threshold for data above the image background
  gaussian_size: 1.0121397464510862 # Gaussian blur intensity in px
  gaussian_mode: nearest
  # Scar remvoal parameters. Be careful with editing these as making the algorithm too sensitive may
  # result in ruining legitimate data.
  remove_scars:
    run: true
    removal_iterations: 2 # Number of times to run scar removal.
    threshold_low: 0.250 # below values make scar removal more sensitive
    threshold_high: 0.666 # below values make scar removal more sensitive
    max_scar_width: 4 # Maximum thichness of scars in pixels.
    min_scar_length: 16 # Minimum length of scars in pixels.
grains:
  run: true # Options : true, false
  # Thresholding by height
  threshold_method: std_dev # Options : std_dev, otsu, absolute
  otsu_threshold_multiplier: 1.0
  threshold_std_dev:
    below: 10.0 # Threshold for grains below the image background
    above: 1.0 # Threshold for grains above the image background
  threshold_absolute:
    below: -1.0 # Threshold for grains below the image background
    above: 1.0 # Threshold for grains above the image background
  # Thresholding by area
  smallest_grain_size_nm2: 50 # Size in nm^2 of tiny grains/blobs (noise) to remove, must be > 0.0
  absolute_area_threshold:
    above: [ 300, 3000 ] # above surface [Low, High] in nm^2 (also takes null)
    below: [ null, null ] # below surface [Low, High] in nm^2 (also takes null)
grainstats:
  run: true # Options : true, false
  edge_detection_method: binary_erosion # Options: canny, binary erosion. Do not change this unless you are sure of what this will do.
  cropped_size: 40.0 # Length (in nm) of square cropped images (can take -1 for grain-sized box)
disordered_tracing:
  run: true # Options : true, false
  min_skeleton_size: 10 # Minimum number of pixels in a skeleton for it to be retained.
  skeletonisation_method: topostats # Options : zhang | lee | thin | topostats
  pad_width: 1 # Cells to pad grains by when tracing
#  cores: 1 # Number of cores to use for parallel processing
plotting:
  run: true # Options : true, false
  save_format: png # Options : see https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html
  pixel_interpolation: null # Options : https://matplotlib.org/stable/gallery/images_contours_and_fields/interpolation_methods.html
  image_set: core  # Options : all, core
  zrange: [null, null]  # low and high height range for core images (can take [null, null]). low <= high
  colorbar: true  # Options : true, false
  axes: true # Options : true, false (due to off being a bool when parsed)
  cmap: nanoscope # Options : nanoscope, afmhot, gwyddion
  mask_cmap: blu # Options : blu, jet_r and any in matplotlib
  histogram_log_axis: false # Options : true, false
  histogram_bins: 200 # Number of bins for histogram plots to use
  dpi: 100 # Dots Per Inch used in figures, if set to "figure" will use Matplotlib default
summary_stats:
  run: true # Whether to make summary plots for output data
  config: null
```

To load the configuration file into Python we use the `read_yaml()` function. This saves the options as a dictionary and
we can access values by the keys. The example below prints out the top-levels keys and then the keys for the `filter`
configuration.

There is also a separate configuration file called `plotting_dictionary` which contains parameters for which data to plot and in what format.

**NB** Python dictionaries have keys which can be considered as the parameter that is to be configured and each key has
an associated value which is the value you wish to set the parameter to.

In [ ]:
config = read_yaml(BASE_DIR.parent / "topostats" / "default_config.yaml")
plotting_dictionary = read_yaml(BASE_DIR.parent / "topostats" / "plotting_dictionary.yaml")
print(f"Top level keys of config.yaml : \n\n {config.keys()}\n")
print(f"Configuration options for Filter : \n\n {config['filter'].keys()}")

You can look at all of the options using the `json` package to "pretty" print the dictionary which makes it easier to
read. Here we print the `filter` section. You can see the options map to those of the `Filter()` class with an
additional `"run": true` which is used when running TopoStats at the command line.

In [ ]:
print(json.dumps(config["filter"], indent=4))

We will use the configuration options we have loaded in processing the `minicircle.spm` image. For convenience we save
each set of options to their own dictionary and remove the `run` entry as this is not required when running TopoStats
interactively.

We also set the `plotting_config["image_set"]` to `all` so that all images can be plotted (there are some internal controls that determine whether images are plotted and returned).


In [ ]:
loading_config = config["loading"]
filter_config = config["filter"]
filter_config.pop("run")
grain_config = config["grains"]
grainstats_config = config["grainstats"]
grainstats_config.pop("run")
disordered_config = config["disordered_tracing"]
disordered_config.pop("run")
plotting_config = config["plotting"]
plotting_config.pop("run")
plotting_config["image_set"] = "all"

## Load Scans

The first step before processing is to load a scan, this extracts the image data to a Numpy array along with the
filepath and the pixel to nanometer scaling parameter which is used to correctly scale the pixels to images. These are
stored in nested dictionaries with one top-level entry for each image that is found.

One of the key fields you may wish to change is the `channel`.

In [ ]:
all_scan_data = LoadScans(image_files, config=config)
all_scan_data.get_data()

# Plot the loaded scan in its raw format
fig, ax = plt.subplots(figsize=(8, 8))
plt.imshow(all_scan_data.image, cmap="afmhot")
plt.show()

Now that we have loaded the data we can start to process it. The first step is filtering the image.

## TopoStats Object

We create a `topostats_object` to assign an image's data to. All data and stats collected on the image after this point will also be stored in here.

In [ ]:
topostats_object = all_scan_data.img_dict["minicircle.spm"]

## Get Output Paths

To ensure consistency throughout the program output paths are defined once and passed through the different processes.

The `topostats_object` is also defined here which holds all the data related to a specific image and the statistics collected from it.

In [ ]:
core_out_path, filter_out_path, grain_out_path, tracing_out_path = get_out_paths(
    image_path=topostats_object.img_path,
    base_dir=all_scan_data.config["base_dir"],
    output_dir=all_scan_data.config["output_dir"],
    filename=topostats_object.filename,
    plotting_config=plotting_config,
)

## Filter Image

Now that we have found some images the first step in processing is to filter them to remove some of the noise. This is
achieved using the `Filters()` class.  There are a number of options that we need to specify which are described in the
table below and also in the [documentation](https://topostats.readthedocs.io/en/dev/topostats.filters.html). 





Once we setup a `Filters` object we can call the different methods that are available for it. There are lots of
different methods that carry out the different steps but for convenience the `filter_image()` method runs all these.

The following section instantiates ("sets up") an object called `filtered_image` of type `Filters` using the first file
found (`image_files[0]`) and the various options from the `filter_config` dictionary.


In [ ]:
filtered_image = Filters(topostats_object=topostats_object, **filter_config)
filtered_image.filter_image()

The `filtered_image` now has a number of NumPy arrays saved in the `.images` dictionary that can be accessed and plotted. To view
the names of the images (technically the dictionary keys) you can print them with `filter_image.images.keys()`...

In [ ]:
print(f"Available NumPy arrays to plot in filter_image.images dictionary :\n\n{filtered_image.images.keys()}")

To plot the raw extracted pixels you can use the built-in NumPy method `imshow()`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
plt.imshow(filtered_image.images["gaussian_filtered"], cmap="afmhot")
plt.show()

TopoStats includes a custom plotting class `Images`  which formats plots in a more familiar manner.

It has a number of options, please refer to the official documentation on
[configuration](https://afm-spm.github.io/TopoStats/configuration.html) under the `plotting` entry for what these values
are or the [API
reference](https://afm-spm.github.io/TopoStats/topostats.plottingfuncs.html#module-topostats.plottingfuncs).

The class requires a Numpy array, which we have just generated many of during the various filtering stages, and a number
of options. Again for convenience we use the `**plotting_config` notation to unpack the key/value pairs stored in the
`plotting_config` dictionary.




In [ ]:
fig, ax = Images(
    data=filtered_image.images["gaussian_filtered"],
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
display(fig)

Here we plot the image after processing and zero-averaging the background but with the `viridis` palette and
constraining the `zrange` to be between 0 and 3

In [ ]:
# First remove the current value for cmap in the plotting_config dictionary, otherwise an error occurs because the same
# argument will have been specified twice.
current_cmap = plotting_config.pop("cmap")
current_zrange = plotting_config.pop("zrange")
fig, ax = Images(
    data=filtered_image.images["gaussian_filtered"],
    filename=filtered_image.filename,
    output_dir="img/",
    cmap="viridis",
    zrange=[0, 3],
    save=True,
    **plotting_config,
).plot_and_save()
# Restore the value for cmap to the dictionary.
plotting_config["cmap"] = current_cmap
plotting_config["zrange"] = current_zrange
fig

## Finding Grains

The next step in processing the image is to find grains - a.k.a the molecules we want to analyse. This is done using the `Grains` class and we have saved the
configuration to the `grains_config` dictionary. For details of the arguments and their values please refer to the
[configuration](https://afm-spm.github.io/TopoStats/configuration.html) and the [API
reference](https://afm-spm.github.io/TopoStats/topostats.grains.html).

The most important thing required for grain finding is the resulting image from the Filtering stage, however several
other key variables are required. Again there is a one-to-one mapping between the options to the `Grains()` class and
their values in the configuration file.

As with `Filters` the `Grains` class has a number of methods that carry out the grain finding, but there is a convenience method
`find_grains()` which calls all these in the correct order.

In [ ]:
plotting_config["run"] = True
plotting_config["plot_dict"] = plotting_dictionary

run_grains(
    topostats_object=topostats_object,
    grain_out_path=grain_out_path,
    core_out_path=core_out_path,
    plotting_config=plotting_config,
    grains_config=grain_config,
)

plotting_config.pop("run")

grains = Grains(topostats_object=topostats_object, **grain_config)
grains.find_grains()

plotting_config.pop("plot_dict")

The `grains` object now also contains a series of images that we can plot.

In [ ]:
print(f"Available NumPy arrays to plot in grains:\n\n{len(topostats_object.grain_crops)}")

And we can again use the `plot_and_save()` function to plot these.

In [ ]:
plotting_config["colorbar"] = False
fig, ax = Images(
    data=grains.image,
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
fig

### Thresholds

The thresholds can be used in different ways based on the direction you want to detect grains. Typically for molecular
imaging where the DNA or protein is raised above the background you will want to look for objects above the surface, using the `above`
threshold. However, when imaging silicon, you may be interested in objects below the surface, using the `below` threshold. For convenience it is
possible to look for grains that are both above the `above` threshold and `below` the below threshold.

If you want to change the option you can update the `config["grains"]` dictionary as we do below.

So far the thresholding method used has been `threshold_method="std_dev"` defined in the configuration file we
loaded. This calculates the mean and standard deviation of height across the whole image and then determines the
threshold by scaling the standard deviation by a given factor (defined by `threshold_std_dev`) and adding it to the mean
to give the `above` threshold and/or subtracting if from the mean to give the `below` threshold.

An alternative method is to use the `threshold_method="absolute"` and explicitly state the
`below` and `above` thresholds (although since we are only looking for objects above a given threshold only the `above`
value will be used). If you wish to specify values for both they are shown below.

In [ ]:
plotting_config["run"] = True
plotting_config["plot_dict"] = plotting_dictionary
grain_config["run"] = True

temp_topostats_object = copy.deepcopy(topostats_object)

grain_config["threshold_method"] = "absolute"
grain_config["threshold_absolute"]["above"] = 0.01  # Change just the above threshold
grain_config["threshold_absolute"]["below"] = []  # Change just the below threshold
grain_config["threshold_absolute"] = {
    "below": [],
    "above": 1.2,
}  # Change both the below and above threshold

temp_topostats_object.image = filtered_image.images["final_zero_average_background"]
temp_topostats_object.filename = filtered_image.filename
temp_topostats_object.pixel_to_nm_scaling = filtered_image.pixel_to_nm_scaling


run_grains(
    topostats_object=temp_topostats_object,
    grain_out_path=grain_out_path,
    core_out_path=core_out_path,
    plotting_config=plotting_config,
    grains_config=grain_config,
)

grains_absolute = Grains(topostats_object=temp_topostats_object, **grain_config)
grains_absolute.find_grains()

This is important because you need to know where the resulting images are stored within the `Grains.direction`
dictionary. This will have entries corresponding to the `direction` that grains have been searched for.

In [ ]:
print(f"Grains available in original 'grains' (std_dev, both) : {len(grains.grain_crops)}")
print(f"Grains available in absolute (absolute, above)        : {len(grains_absolute.grain_crops)}")

## Grain Statistics

Now that the grains have been found we can calculate statistics for each. This is done using the `GrainStats()`
class. Again the configuration options from the YAML file map to those of the class and there is a convenience method
`calculate_stats()` which will run all steps of grain finding. However, because the class is processing results that we
have generated we have to explicitly pass in more values.

For details of what the arguments are please refer to the [API
reference](https://afm-spm.github.io/TopoStats/topostats.grainstats.html).

The `GrainStats` class returns a Pandas `pd.DataFrame` of calculated statistics We therefore instantiate ("set-up") the `grain_stats` dictionary
to hold these results.



In [ ]:
grainstats = GrainStats(topostats_object=topostats_object, base_output_dir="grains")
grainstats.calculate_stats()
grain_stats = {crop_id: crop.stats for crop_id, crop in grainstats.grain_crops.items()}

`grain_stats` is a dictionary. We can print this out as
shown below.

In [ ]:
grain_stats

### Plotting Individual Grains

It is possible to plot the individual grains in the same way that whole images are plotted. These are created when running `Grains()` earlier in this notebook and are stored in `[output_dir]/processed/[filename]/` after a successful run of TopoStats. Different types of plots can be found in this directory, here we will access an ordered trace of a single grain (stored in `[output_dir]/processed/[filename]/dnatracing/ordered/`).

The naming convention for the saved `.png` files is `[filename]_grain_[grain_id]_ordered_traces`, here we will view the first grain (id 0). We use an `f string` to mark a space to automatically replace with the image's filename within a regular string. 

In [ ]:
img_file = (
    Path(BASE_DIR.parent)
    / config["output_dir"]
    / "processed"
    / topostats_object.filename
    / "dnatracing"
    / "ordered"
    / f"{topostats_object.filename}_grain_0_ordered_traces.png"
)
ordered_trace_img = Image.open(img_file)
fig, ax = plt.subplots()
ax.imshow(ordered_trace_img)
ax.axis("off")
plt.show()

## DNA Tracing

When working with molecules it is possible to calculate DNA Tracing Statistics using the `disordered_tracing.py`'s `disorderedTrace.trace_dna()` function which takes an image and grain masks, and returns statistics about the dna.

Here we select a single grain and print the statistics for it.

In [ ]:
trace_image_disordered(
    topostats_object=grains.topostats_object,
    **disordered_config,
)

tracing_results = topostats_object.grain_crops[2].disordered_trace.stats

print(tracing_results)

These statistics can now be plotted to show the distribution of the different metrics. Please see the Jupyter Notebook
`notebooks/02-Summary-statistics-and-plots.ipynb` for examples of how to plot these statistics.